# Accuracy evaluation on Colab (Precision@5)

Runs the frozen evaluation from the README with the repo's local model
(`qwen3:4b-instruct-2507-q4_K_M` through Ollama) on a Colab GPU:

1. **Pilot:** 8 paired runs on dev resumes R01–R02. Every run must succeed before evaluation can start.
2. **Main:** 96 runs, with 8 held-out resumes (R03–R10) × 2 architectures × 20/50 jobs × 3 repetitions.
3. **Score:** Precision@5 against `data/labels/reference_labels.csv`, plus latency, tokens, and the project's decision rule.

**Before you start:** pick **Runtime → Change runtime type → GPU**. An L4 or A100 is fastest; a free T4 works too.
Then use **Runtime → Run all**. Results are saved to Google Drive after every run.
If Colab disconnects, reconnect and **Run all** again. The runner skips runs that are already recorded.

Expect roughly 4–6 hours on a T4 and less on an L4/A100. The context cache is stored as 8-bit (`q8_0`)
so the 131k-token window used by `shared/local_experiment.json` fits in GPU memory. Pilot and main
both use these same settings.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: choose Runtime > Change runtime type > GPU"

In [ ]:
# Keep results on Google Drive so a disconnect does not lose completed runs.
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/agentic_resume_matcher_results'
PILOT, MAIN = f'{RESULTS}/pilot', f'{RESULTS}/main'
!mkdir -p "{RESULTS}"

In [ ]:
%cd /content
!test -d agentic_resume_matcher || git clone https://github.com/RitvikNeerattil/agentic_resume_matcher.git
%cd /content/agentic_resume_matcher
# Pin the code version on first start so resumed sessions run identical code.
import os
pin = f'{RESULTS}/code_commit.txt'
!git fetch -q origin
if os.path.exists(pin):
    commit = open(pin).read().strip()
else:
    commit = !git rev-parse origin/main
    commit = commit[0]
    open(pin, 'w').write(commit)
!git checkout -q {commit} && git log -1 --format="Code version: %h %s"

In [ ]:
# Install and start Ollama (flash attention + 8-bit context cache to fit 131k tokens on the GPU).
import os, subprocess, time, urllib.request
!command -v ollama >/dev/null || (apt-get -qq install -y zstd >/dev/null && curl -fsSL https://ollama.com/install.sh | sh)
env = {**os.environ, 'OLLAMA_FLASH_ATTENTION': '1', 'OLLAMA_KV_CACHE_TYPE': 'q8_0', 'OLLAMA_NUM_PARALLEL': '1'}
subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/version'); print('Ollama is running'); break
    except Exception:
        time.sleep(1)
!ollama pull qwen3:4b-instruct-2507-q4_K_M

In [ ]:
# Offline tests, then a one-token request at the full context size to confirm it fits on the GPU.
!python -m unittest shared.test_workflow
import json, urllib.request
body = {'model': 'qwen3:4b-instruct-2507-q4_K_M', 'prompt': 'hi', 'stream': False,
        'options': {'num_ctx': 131072, 'num_predict': 1}}
urllib.request.urlopen(urllib.request.Request('http://127.0.0.1:11434/api/generate', json.dumps(body).encode(),
                                              {'Content-Type': 'application/json'}), timeout=600).read()
!ollama ps
print('PROCESSOR should read "100% GPU". A CPU/GPU split still works, but more slowly.')

## 1. Pilot (8 runs on R01–R02)

In [ ]:
!python -m shared.run_experiment --phase pilot --architectures both --config shared/local_experiment.json --output "{PILOT}" --execute --resume
import json
runs = [json.loads(l) for l in open(f'{PILOT}/runs.jsonl')]
for r in runs:
    print(r['resume_id'], r['size'], f"{r['architecture']:12}", r['status'], f"{r['seconds']:.0f}s")
failed = [r for r in runs if r['status'] != 'ok']
assert len(runs) == 8 and not failed, ('Pilot must fully succeed before evaluation. Rename or delete the pilot '
                                       'folder in Drive and run this cell again, or share runs.jsonl with the team.')
print('Pilot passed: all 8 runs succeeded')

## 2. Main evaluation (96 runs on held-out R03–R10)

In [ ]:
!python -m shared.run_experiment --phase main --architectures both --config shared/local_experiment.json --pilot-dir "{PILOT}" --output "{MAIN}" --execute --resume

## 3. Accuracy (Precision@5), latency and tokens

In [ ]:
!python -m shared.summarize_results "{MAIN}" > /dev/null
import pandas as pd, json
table = pd.read_csv(f'{MAIN}/comparison.csv')
cols = ['architecture', 'size', 'runs', 'precision_at_5', 'precision_sd', 'failure_rate', 'median_seconds',
        'input_tokens_known', 'output_tokens_known']
display(table[cols].round(3))
display(pd.read_csv(f'{MAIN}/paired_resumes.csv').round(3))
print(json.load(open(f'{MAIN}/decision.json')))

In [ ]:
# Download the summaries (no raw model responses) to add to data/reports/.
import shutil, os
out = '/content/accuracy_summary'
os.makedirs(out, exist_ok=True)
for name in ('comparison.csv', 'paired_resumes.csv', 'decision.json', 'batch.json'):
    if os.path.exists(f'{MAIN}/{name}'):
        shutil.copy(f'{MAIN}/{name}', out)
shutil.make_archive('/content/accuracy_summary', 'zip', out)
from google.colab import files
files.download('/content/accuracy_summary.zip')
print('The full raw runs stay in Google Drive:', RESULTS)